# 04 - The LangGraph agent

**Goal:** show the agent's graph, its tools and two real questions: an interaction question
answered with citations, and a dose question refused. The agent is the app's own
(`src/agent/`), with Ollama (qwen2.5:3b, bge-m3) and the Qdrant index. The medicine tables are
loaded from `data/processed/pharma` into a temporary SQLite file, outside the repository.

In [1]:
import os
import sys
import warnings
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
ROOT = Path.cwd().resolve().parents[1]  # notebooks/<module>/ -> repository root
for path in (ROOT, ROOT / "backend"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(ROOT)
warnings.filterwarnings("ignore")

import logging
import tempfile

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from PIL import Image

plt.rcParams.update({"figure.dpi": 72, "savefig.dpi": 72})
%config InlineBackend.figure_formats = ["jpeg"]
pd.set_option("display.max_colwidth", 90)


def fmt(frame, digits=3, **columns):
    """Format the float columns with f"{x:.{digits}f}", the rounding of the result tables."""
    out = frame.copy()
    for name in out.columns:
        if out[name].dtype.kind == "f":
            d = columns.get(name, digits)
            out[name] = out[name].map(lambda v, d=d: "" if pd.isna(v) else f"{v:.{d}f}")
    return out


def show_image(path, title="", width=480):
    """Display an image file, downscaled (the notebook stays small)."""
    import io

    from IPython.display import Image as JpegImage

    image = Image.open(path).convert("RGB")
    image.thumbnail((width, width))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=80)
    if title:
        display(Markdown(f"*{title}*"))
    display(JpegImage(data=buffer.getvalue(), format="jpeg"))

In [2]:
from app.config import Settings
from app.db import make_engine
from app.pharma_loader import Inputs, load
from src.agent.runtime import AgentHolder

pharma_file = Path(tempfile.mkdtemp(prefix="tujya-pharma-")) / "pharma.db"
engine = make_engine(f"sqlite:///{pharma_file.as_posix()}")
print(load(engine, Inputs.default()))
settings = Settings(rag_model_cache_dir=Path(tempfile.gettempdir()) / "tujya-fastembed")
holder = AgentHolder(settings, engine)
agent = holder.agent()
for name in ("httpx", "httpcore", "huggingface_hub"):
    logging.getLogger(name).setLevel(logging.WARNING)

LoadReport(pills=380, ingredients=1793, pill_ingredients=417, interactions=75653)


## 1. The graph

`get_graph().draw_mermaid()` of the compiled LangGraph graph.

In [3]:
mermaid = agent.graph.get_graph().draw_mermaid()
print(mermaid)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	understand(understand)
	tools(tools)
	retrieve(retrieve)
	answer(answer)
	safety(safety)
	__end__([<p>__end__</p>]):::last
	__start__ --> understand;
	answer --> safety;
	retrieve --> answer;
	tools --> retrieve;
	understand -.-> safety;
	understand -.-> tools;
	safety --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. The tools

The same six tools are given to the LLM (LangChain tools) and to MCP clients
(`src/agent/mcp_server.py`).

In [4]:
from src.agent.tools import TOOL_SPECS

pd.DataFrame([{"tool": name, "description": spec[2][:110]} for name, spec in TOOL_SPECS.items()])

,tool,description
0,identify_pills,Detect the pills on a photo and identify each one (top-5 with probabilities). Identifi...
1,check_interaction,"Check DDInter for interactions between two or more drugs or pill classes: severity, de..."
2,get_drug_info,Pill classes containing an active ingredient and excerpts of its openFDA label (no dos...
3,search_documents,"Search openFDA labels, PubMed abstracts and DDInter rows (bge-m3 embeddings, BM25 or h..."
4,lookup_brand,"Active ingredients of a medicine brand (Panadol, Doliprane, Efferalgan...), with the s..."
5,predict_toxicity,Only for questions about toxicity: probability that each molecule (ingredient name or ...


In [5]:
from src.agent.mcp_server import build_server

server = build_server(holder.toolbox())
tools = await server.list_tools()
print("MCP tools:", [t.name for t in tools])

MCP tools: ['identify_pills', 'check_interaction', 'get_drug_info', 'search_documents', 'lookup_brand', 'predict_toxicity']


## 3. An interaction question

The answer cites numbered sources; the safety node checks every sentence.

In [6]:
def show(result):
    display(Markdown(result["answer"]))
    display(
        pd.DataFrame(
            [
                {
                    "n": i + 1,
                    "source": s.get("source_name"),
                    "type": s.get("doc_type"),
                    "title": str(s.get("title"))[:70],
                }
                for i, s in enumerate(result["sources"])
            ]
        )
    )
    print("tools called:", [c["name"] for c in result.get("tool_calls", [])])
    print("safety:", result.get("safety", {}).get("action"))


show(await agent.run("Does warfarin interact with fluconazole?"))

[1] [2] DDInter 2.0 lists an interaction between fluconazole and warfarin with severity Major. The Fluconazole label mentions that prothrombin time may be increased in patients receiving concomitant fluconazole and coumarin-type anticoagulants, such as warfarin [2]. Careful monitoring of prothrombin time is recommended for these patients [2].

Information uniquement, consultez un pharmacien ou un médecin.

,n,source,type,title
0,1,DDInter 2.0 drug-drug interactions,interaction,DDInter 2.0: warfarin + fluconazole (Major)
1,2,openFDA drug label,label,Fluconazole label (Fluconazole): Drug interactions
2,3,DDInter 2.0,interaction,DDInter 2.0: fluconazole + warfarin (Major)
3,4,openFDA drug label,label,Fluconazole label (Fluconazole): Precautions
4,5,PubMed (Biol Pharm Bull),abstract,Comparison of the effects of azole antifungal agents on the anticoagul
5,6,openFDA drug label,label,Warfarin label (Warfarin Sodium): Drug interactions


tools called: ['check_interaction', 'search_documents']
safety: ok


## 4. A dose question

Refused before any LLM call: the app never gives a dose.

In [7]:
show(await agent.run("How many mg of warfarin should I take per day?"))

Tujya ne donne jamais de dose ni de posologie. Pour savoir quelle quantité prendre, demandez à votre pharmacien ou à votre médecin : ils tiennent compte de votre situation.

Information uniquement, consultez un pharmacien ou un médecin.

""


tools called: []
safety: refused
